In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# SoH forecast model comparison — XGBoost vs Chronos-2 vs TimesFM-2.5 vs Persistence

Fair per-vehicle backtest: hold out each vehicle's last ~40% of monthly SoH and forecast it from
history only. **XGBoost** = feature-based condition-aware rate model (leave-one-vehicle-out,
rolled forward with persisted recent stress). **Chronos-2 / TimesFM-2.5** = zero-shot univariate
forecasters on the SoH series. **Persistence** = naive last-value baseline.

Produces: aggregate + degrading/flat-split RMSE, a bar chart, and **per-vehicle prediction plots**.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import model, xgboost as xgb
from chronos import BaseChronosPipeline
from timesfm import TimesFM_2p5_200M_torch, ForecastConfig

m = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin', 'month'])
tr_all = model.build_transitions(m)
FEATS, STATE, STRESS = model.FEATS, model.STATE, model.STRESS

def train_xgb(tr):
    return xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8,
                            colsample_bytree=0.8, n_jobs=8, verbosity=0).fit(
        tr[FEATS].to_numpy(), tr['loss'].to_numpy(), sample_weight=tr['w'].to_numpy())

def xgb_sim(ctx, mdl, H):
    g = ctx.sort_values('month'); last = g.iloc[-1]; stress = g.iloc[-6:][STRESS].median().to_dict()
    st = {s: last[s] for s in STATE}; soh = last['soh']; out = []
    for _ in range(H):
        isa, dfc = model._curv(st['age_months'], st['soh'])
        x = pd.DataFrame([{**{s: st[s] for s in STATE}, **stress, 'inv_sqrt_age': isa, 'soh_deficit': dfc}])[FEATS].to_numpy()
        soh = soh - max(mdl.predict(x)[0], 0)
        st.update(soh=soh, age_months=st['age_months'] + 1, cum_ah=st['cum_ah'] + stress.get('ah_throughput', 0))
        out.append(soh)
    return np.array(out)

pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-base', device_map='cuda', torch_dtype=torch.bfloat16)
def chronos(ctx, H):
    q, _ = pipe.predict_quantiles(torch.tensor(ctx, dtype=torch.float32), prediction_length=H, quantile_levels=[0.5])
    return q[0, :, 0].cpu().float().numpy()

tfm = TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(ForecastConfig(max_context=256, max_horizon=64, normalize_inputs=True, use_continuous_quantile_head=True))
def timesfm_f(ctx, H):
    pf, _ = tfm.forecast(horizon=H, inputs=[np.asarray(ctx, dtype=float)]); return np.asarray(pf[0])[:H]
print('models loaded')

## 1. Backtest — forecast each vehicle's held-out tail

In [ ]:
MODS = ['XGBoost', 'Chronos-2', 'TimesFM-2.5', 'Persistence']
runs = []
for vin, g in m.groupby('vin'):
    g = g.sort_values('month').reset_index(drop=True); n = len(g)
    if n < 10:
        continue
    split = int(np.ceil(0.6 * n))
    if n - split < 2:
        continue
    ctx, test = g.iloc[:split], g.iloc[split:]; act = test['soh'].to_numpy(); H = len(test)
    xm = train_xgb(tr_all[tr_all['vin'] != vin])
    preds = {'XGBoost': xgb_sim(ctx, xm, H), 'Chronos-2': chronos(ctx['soh'].to_numpy(), H),
             'TimesFM-2.5': timesfm_f(ctx['soh'].to_numpy(), H), 'Persistence': np.full(H, ctx['soh'].iloc[-1])}
    runs.append(dict(vin=vin, g=g, split=split, act=act, preds=preds,
                     tail_decline=g['soh'].iloc[split - 1] - g['soh'].iloc[-1]))
res = pd.DataFrame([{'vin': r['vin'], 'H': len(r['act']), 'tail_decline': r['tail_decline'],
                     **{f'{k}_rmse': np.sqrt(np.mean((np.asarray(v)[:len(r['act'])] - r['act'])**2)) for k, v in r['preds'].items()}}
                    for r in runs])
res.to_csv('data/mahindra/soh/model_comparison.csv', index=False)
print(f'{len(runs)} vehicles backtested (forecast last ~40% of each series)')
for label, mask in [('ALL', res['vin'].notna()), ('DEGRADING (>=2pp)', res['tail_decline'] >= 2), ('FLAT (<2pp)', res['tail_decline'] < 2)]:
    sub = res[mask]
    print(f"\n{label} (n={len(sub)}) mean RMSE:  " + '  '.join(f'{mm} {sub[mm+"_rmse"].mean():.2f}' for mm in MODS))

## 2. RMSE by model — degrading vs flat tail

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (label, mask) in zip(axes, [('Degrading tail (>=2pp)', res['tail_decline'] >= 2), ('Flat tail (<2pp)', res['tail_decline'] < 2)]):
    sub = res[mask]; vals = [sub[mm + '_rmse'].mean() for mm in MODS]
    ax.bar(MODS, vals, color=['C0', 'C1', 'C4', 'grey']); ax.set_title(f'{label}  (n={len(sub)})')
    ax.set_ylabel('mean RMSE (pp)'); ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
fig.suptitle('SoH forecast backtest — RMSE by model', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Per-vehicle prediction visualization

Actual SoH (blue, with the context|forecast split marked) vs each model's forecast of the held-out
tail. Shown for the most-degrading vehicles (where the models actually differ).

In [ ]:
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
# Forward forecast to 5 years of age (XGBoost trained on ALL vehicles)
xgb_full = train_xgb(tr_all)
WARRANTY_YR = 5.0
show = sorted(runs, key=lambda r: -r['tail_decline'])[:12]
n=len(show); ncol=4; nrow=int(np.ceil(n/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(4.9*ncol,3.5*nrow),squeeze=False)
for ax in axes.flat: ax.axis('off')
for i,r in enumerate(show):
    ax=axes[i//ncol][i%ncol]; ax.axis('on'); g=r['g'].sort_values('month'); mth=g['month']
    reg = mth.iloc[0] - pd.Timedelta(days=float(g['age_months'].iloc[0])*30.4375)
    last_age=float(g['age_months'].iloc[-1])
    H=max(int(round(WARRANTY_YR*12 - last_age)),1)
    fc=xgb_sim(g, xgb_full, H)
    fut=[mth.iloc[-1]+pd.Timedelta(days=30.4375*k) for k in range(1,H+1)]
    ax.plot(mth, g['soh'], 'o-', color='C0', ms=3, lw=1.3, label='actual')
    ax.plot(fut, fc, '--', color='C2', lw=2, label='XGBoost forecast')
    we=reg+pd.DateOffset(years=int(WARRANTY_YR))
    ax.axvline(reg, ls='-', color='0.45', lw=1.1, label='registration (SoH=100)')
    ax.plot([reg], [100], marker='v', color='0.45', ms=6, clip_on=False, zorder=5)
    ax.axvline(we, ls='-.', color='green', lw=1.2, label='5-yr warranty')
    ax.axhline(80, ls=':', color='red', lw=0.8)
    soh5=float(fc[-1])
    ax.set_title(f"{r['vin'][-8:]}  reg {reg:%b'%y}  now {float(g['soh'].iloc[-1]):.0f}% -> 5yr ~{soh5:.0f}%", fontsize=8)
    ax.set_xlim(reg-pd.Timedelta(days=25), we+pd.Timedelta(days=45))
    ax.set_ylim(min(float(g['soh'].min()), float(np.min(fc)), 78)-1, 101)
    ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
    rn=mdates.date2num(reg)
    secax=ax.secondary_xaxis('top', functions=(lambda x, rn=rn:(np.asarray(x)-rn)/365.25,
                                               lambda y, rn=rn: rn+np.asarray(y)*365.25))
    secax.xaxis.set_major_locator(MultipleLocator(1)); secax.tick_params(labelsize=6)
    secax.set_xlabel('age (years)', fontsize=6, labelpad=1)
    if i==0: ax.legend(fontsize=6, loc='lower left')
fig.suptitle('Per-vehicle SoH: actual + XGBoost forecast to 5 years from registration  (grey = registration / SoH 100, top axis = age in years)', fontsize=12)
plt.tight_layout(rect=[0,0,1,0.95]); plt.show()